# Atelier 02 — RAG Simple FAISS

**Objectif** : construire un pipeline RAG **FAISS-only** de bout en bout sur les documents du logement.

**Sujets couverts** :
1. Extraction de texte PDF (PyMuPDF)
2. 3 stratégies de chunking comparées
3. Génération d'embeddings multilingues
4. Index vectoriel FAISS + recherche par similarité
5. Pipeline RAG complet (LCEL)
6. Évaluation Recall@k

> **Note** : ChromaDB est introduit à l'atelier 03 (pipeline + agent). Ici on se concentre sur FAISS qui est la base la plus simple/rapide.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

docs_dir = "../data/documents"
pdfs = [f for f in os.listdir(docs_dir) if f.endswith('.pdf')]
print(f"PDFs disponibles : {len(pdfs)}")
for f in pdfs:
    size = os.path.getsize(os.path.join(docs_dir, f)) / 1024
    print(f"  - {f} ({size:.1f} Ko)")

## 1. Extraction de texte PDF

**Piège** : `pymupdf` s'installe via `pip install pymupdf` mais s'importe `import fitz`.

In [ ]:
import fitz  # pymupdf

path = "../data/documents/notice_chaudiere.pdf"
doc = fitz.open(path)
print(f"Nombre de pages : {len(doc)}")
print("\n=== Page 1 (extrait) ===")
print(doc[0].get_text()[:500] + "...")
doc.close()

## 2. Trois stratégies de chunking comparées

Concept RAG : le chunking impacte ~80 % de la qualité d'un RAG. Trop petit → on perd le contexte. Trop grand → on noie le LLM dans du bruit.

1. **Fixed-size** (caractères) — simple, mais coupe les phrases
2. **Recursive** par séparateurs (\n\n → \n → . → espace) — recommandé par défaut
3. **Semantic** (rupture sur similarité cosinus) — plus lent mais cohérent

In [ ]:
from homebutler.rag.ingestion import (
    load_pdf_with_metadata, chunk_fixed_size, chunk_recursive, ingest_all_documents
)

docs_pages = []
for f in os.listdir('../data/documents'):
    if f.endswith('.pdf'):
        docs_pages.extend(load_pdf_with_metadata(os.path.join('../data/documents', f)))
print(f"Pages chargées : {len(docs_pages)}")

chunks_fixed     = chunk_fixed_size(docs_pages, chunk_size=512, chunk_overlap=50)
chunks_recursive = chunk_recursive(docs_pages, chunk_size=512, chunk_overlap=50)

import numpy as np
def analyze(chunks, label):
    sizes = [len(c.page_content) for c in chunks]
    print(f"\n{label:20s}  n={len(chunks):3d}  moy={np.mean(sizes):5.0f}  min={min(sizes):3d}  max={max(sizes):4d}")

analyze(chunks_fixed, 'Fixed-size 512c')
analyze(chunks_recursive, 'Recursive 512c')

## 3. Embeddings multilingues

Modèle : `paraphrase-multilingual-MiniLM-L12-v2`
- 50 langues (dont français)
- Dimension 384 (vs 1536 pour OpenAI text-embedding-3-small)
- ~120 Mo, tourne en CPU

Concept : deux textes proches sémantiquement → vecteurs proches géométriquement (similarité cosinus ≈ 1).

In [ ]:
from homebutler.rag.vectorstore_faiss import get_embeddings

embeddings = get_embeddings()

test_queries = [
    "température chaudière nuit",
    "chauffe-eau réglage nocturne",       # synonyme
    "boiler night temperature setting",   # anglais — modèle multilingue
]
vecs = embeddings.embed_documents(test_queries)
print(f"Dimension : {len(vecs[0])}")

from numpy import dot
from numpy.linalg import norm
def cosine(a, b): return dot(a, b) / (norm(a) * norm(b))
print(f"\ncos(fr1, fr2) = {cosine(vecs[0], vecs[1]):.3f}  (synonymes FR)")
print(f"cos(fr1, en)  = {cosine(vecs[0], vecs[2]):.3f}  (cross-lingual)")

## 4. Indexation FAISS + recherche

FAISS = bibliothèque ANN (Approximate Nearest Neighbors) de Meta.
Pour 10k documents, FAISS retrouve les top-k en quelques ms (vs secondes en force brute).

In [ ]:
import time
from homebutler.rag.vectorstore_faiss import build_faiss_index

t0 = time.time()
faiss_store = build_faiss_index(
    chunks_recursive,
    save_path="../data/faiss_index",
    force_rebuild=True,
)
print(f"\nFAISS construit en {time.time()-t0:.1f}s")

In [ ]:
query = "température recommandée chaudière la nuit"
t0 = time.time()
hits = faiss_store.similarity_search(query, k=3)
print(f"Recherche en {(time.time()-t0)*1000:.1f}ms\n")
for i, h in enumerate(hits, 1):
    print(f"=== Résultat {i} — {h.metadata.get('source')} p.{h.metadata.get('page')} ===")
    print(h.page_content[:200], "...\n")

## 5. Pipeline RAG complet (LCEL)

LCEL (LangChain Expression Language) = composition de runnables avec `|`. Plus transparent que `RetrievalQA`.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from homebutler.llm.provider import get_llm
from homebutler.llm.prompts import RAG_QA_TEMPLATE

retriever = faiss_store.as_retriever(search_type="mmr", search_kwargs={"k": 4})
llm = get_llm(temperature=0.1)

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata.get('source')} p.{d.metadata.get('page')}]\n{d.page_content}" for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_QA_TEMPLATE | llm | StrOutputParser()
)

print(rag_chain.invoke("Quelle est la marque de ma chaudière ?"))

## 6. Évaluation Recall@k

**Recall@k** = sur N questions étalons, combien obtiennent au moins un chunk pertinent dans les k premiers résultats.

Aligné sur le Slide 8 Chapitre 2 (Métriques RAGAs).

In [ ]:
BENCHMARK = [
    ("Température chaudière nuit",        "notice_chaudiere.pdf"),
    ("Code erreur F4 chaudière",          "notice_chaudiere.pdf"),
    ("Purge radiateurs procédure",        "notice_chaudiere.pdf"),
    ("Entretien annuel chaudière",        "notice_chaudiere.pdf"),
    ("Filtre VMC nettoyage",              "notice_vmc.pdf"),
]

def recall_at_k(k):
    hits = 0
    for q, expected_src in BENCHMARK:
        results = faiss_store.similarity_search(q, k=k)
        if any(r.metadata.get("source") == expected_src for r in results):
            hits += 1
    return hits / len(BENCHMARK)

for k in (1, 3, 5):
    print(f"Recall@{k} = {recall_at_k(k):.0%}")